# 07C — Run Benchmark-Locked 3 Evaluator Agents × 159 — FINAL V2

Panel dari benchmark FINAL V5:
1. Gemini 3.1 Pro Preview
2. GPT-5.2
3. Claude Sonnet 4.6

Total target valid evaluation: **159 × 3 = 477**.

Protokol API dibuat konsisten dengan benchmark:
- prompt/rubrik **identik** dengan benchmark V5;
- prompt hash harus `d0e3d1938a7dcd05`;
- provider dipatok;
- provider fallback `False`;
- `max_tokens=2048`;
- **temperature tidak dikirim**, sama seperti benchmark;
- tidak menggunakan `response_format`, karena benchmark menggunakan output JSON melalui instruksi + parser;
- reference test, skenario, chrF++, BLEU, S3E, dan MBR utility tidak masuk evaluator.

Response yang secara teknis tidak dapat diparse/schema-invalid/logically-invalid dicatat dan dapat dicoba ulang secara terbatas. Voting hanya memakai satu response valid final per `(eval_case_id, evaluator)`.

## 1. Environment + locked panel + OpenRouter key

In [1]:
from __future__ import annotations

import getpass
import hashlib
import json
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from tqdm.auto import tqdm

PROJECT_ROOT = Path(
    os.getenv(
        "SKRIPSI_PROJECT_ROOT",
        "/workspace/10122233_CikitaMeydilvira_Skripsi",
    )
).expanduser().resolve()

LOCK_PATH = (
    PROJECT_ROOT
    / "Config"
    / "evaluator_panel_locked.json"
)

PREP_DIR = (
    PROJECT_ROOT
    / "Hasil"
    / "llm_evaluator_main"
    / "prepared"
)

BLIND_PATH = (
    PREP_DIR
    / "main_evaluator_blind_159.csv"
)

PREP_MANIFEST_PATH = (
    PREP_DIR
    / "main_evaluator_preparation_manifest.json"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "Hasil"
    / "llm_evaluator_main"
    / "panel_runs"
)

OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_CHAT_URL = (
    OPENROUTER_BASE_URL
    + "/chat/completions"
)

APP_TITLE = (
    "Skripsi Main Evaluator Bahasa Duri"
)

MAX_TOKENS = 2048
TEMPERATURE_SENT = False

REQUEST_TIMEOUT_SECONDS = 300
MAX_HTTP_RETRIES = 6
BASE_RETRY_DELAY_SECONDS = 2.0
SLEEP_BETWEEN_CALLS_SECONDS = 0.20

# Technical retry only when response cannot be used as a valid structured judgment.
MAX_TECHNICAL_VALIDATION_ATTEMPTS = 3

if not LOCK_PATH.exists():
    raise FileNotFoundError(
        "evaluator_panel_locked.json belum ada. "
        "Jalankan 07B FINAL V2."
    )

if not BLIND_PATH.exists():
    raise FileNotFoundError(
        "main_evaluator_blind_159.csv belum ada. "
        "Jalankan 07B FINAL V2."
    )

lock = json.loads(
    LOCK_PATH.read_text(
        encoding="utf-8"
    )
)

panel = lock.get(
    "selected_panel",
    [],
)

if len(panel) != 3:
    raise RuntimeError(
        f"Locked panel harus 3, ditemukan {len(panel)}."
    )

expected_models = {
    "google/gemini-3.1-pro-preview",
    "openai/gpt-5.2",
    "anthropic/claude-sonnet-4.6",
}

if {
    x["model_id"]
    for x in panel
} != expected_models:
    raise RuntimeError(
        "Locked panel tidak sama dengan Top-3 benchmark V5."
    )

blind_df = pd.read_csv(
    BLIND_PATH,
    dtype={"eval_case_id": str},
    keep_default_na=False,
    encoding="utf-8-sig",
)

required_blind = {
    "eval_case_id",
    "source_text",
    "candidate_text",
    "shared_evidence",
}

if set(blind_df.columns) != required_blind:
    raise RuntimeError(
        f"Blind columns tidak tepat: {list(blind_df.columns)}"
    )

if len(blind_df) != 159:
    raise RuntimeError("Blind input harus 159.")

prep_manifest = json.loads(
    PREP_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

if prep_manifest.get("reference_read") is not False:
    raise RuntimeError(
        "Preparation manifest tidak reference-free."
    )

OPENROUTER_API_KEY = os.getenv(
    "OPENROUTER_API_KEY",
    "",
).strip()

if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass.getpass(
        "OPENROUTER_API_KEY: "
    ).strip()

if not OPENROUTER_API_KEY:
    raise RuntimeError(
        "OPENROUTER_API_KEY kosong."
    )

print("Cases:", len(blind_df))
display(pd.DataFrame(panel))

Cases: 159


,panel_position,evaluator_key,display_name,model_id,developer,family,access_type,provider_only,mean_macro_f1_error_type,std_macro_f1_error_type,error_pairwise_agreement,mean_joint_exact_match,mean_logical_valid_rate
0,1,E3,Gemini 3.1 Pro Preview,google/gemini-3.1-pro-preview,Google,Gemini,Proprietary,[google-ai-studio],0.9121,0.0231,0.9497,0.7107,1.0000
1,2,E1,GPT-5.2,openai/gpt-5.2,OpenAI,GPT,Proprietary,[openai],0.6936,0.0209,0.8742,0.6101,1.0000
2,3,E2,Claude Sonnet 4.6,anthropic/claude-sonnet-4.6,Anthropic,Claude,Proprietary,[anthropic],0.6631,0.0253,0.8931,0.5849,0.9937


## 2. Prompt EXACT dari benchmark FINAL V5

In [2]:
ERROR_TYPES = [
    "correct",
    "meaning_shift",
    "missing_information",
    "added_information",
    "hallucination",
    "untranslated",
    "empty_output",
    "fluency_issue",
    "source_leakage",
]

SEVERITIES = [
    "none",
    "minor",
    "major",
    "critical",
]

VERDICTS = [
    "layak",
    "perlu_ditinjau",
]

SYSTEM_PROMPT = 'Anda adalah Evaluator Agent untuk menilai kualitas terjemahan Bahasa Duri ke Bahasa Indonesia.\n\nINPUT YANG BOLEH DIGUNAKAN\n1. Source Bahasa Duri.\n2. Shared evidence berupa pasangan Duri-Indonesia yang diambil dari korpus latih.\n3. Candidate translation.\n\nShared evidence adalah bukti pendukung, BUKAN reference/gold untuk kasus yang sedang dinilai. Anda tidak menerima reference Indonesia atau gold label kasus.\nPilih tepat satu error_type yang paling dominan.\n\nERROR_TYPE\n- correct: makna dan informasi penting terjaga, bahasa Indonesia dapat diterima, tidak ada kesalahan dominan.\n- meaning_shift: makna utama bergeser, berubah, terbalik, atau hubungan semantis penting berubah.\n- missing_information: informasi penting dari source hilang/tidak diterjemahkan, sementara bagian tersisa masih berupa terjemahan Indonesia.\n- added_information: ada informasi tambahan yang tidak didukung, tetapi belum merupakan halusinasi berat yang mengganti inti makna.\n- hallucination: isi substansial tidak didukung source/evidence, sangat tidak berkaitan, atau mengganti makna utama dengan informasi rekaan.\n- untranslated: sebagian/seluruh unsur Bahasa Duri yang seharusnya diterjemahkan tetap menjadi isi kandidat.\n- empty_output: kandidat kosong atau tidak menghasilkan terjemahan yang dapat dinilai.\n- fluency_issue: makna utama relatif terjaga, tetapi bahasa Indonesia tidak alami, tidak gramatikal, sangat kaku, atau sulit dibaca.\n- source_leakage: terjemahan Indonesia tersedia, tetapi source Bahasa Duri/instruksi/prompt ikut tercetak.\n\nSEVERITY\n- none: hanya untuk correct.\n- minor: kesalahan terbatas dan tidak mengubah informasi utama secara substantif.\n- major: kesalahan memengaruhi informasi penting atau kegunaan terjemahan.\n- critical: keluaran kosong/tidak dapat digunakan, halusinasi berat, atau perubahan makna sangat serius.\n\nVERDICT\n- layak: dapat digunakan tanpa perbaikan substantif.\n- perlu_ditinjau: perlu pemeriksaan/perbaikan sebelum digunakan.\n\nATURAN KONSISTENSI\n- correct -> severity=none dan verdict=layak.\n- selain correct -> severity tidak boleh none.\n- severity major/critical -> verdict=perlu_ditinjau.\n- empty_output -> severity=critical dan verdict=perlu_ditinjau.\n\nBATASAN\n- Jangan memperbaiki/menulis ulang kandidat.\n- Jangan memberi skor numerik.\n- Jangan menebak apakah kandidat berasal dari Zero-Shot, FSP, atau LoRA.\n- Jangan menggunakan reference/gold yang tidak diberikan.\n- reason maksimal dua kalimat.\n\nKeluarkan HANYA satu object JSON tanpa markdown/code fence/teks tambahan:\n{\n  "error_type": "correct|meaning_shift|missing_information|added_information|hallucination|untranslated|empty_output|fluency_issue|source_leakage",\n  "severity": "none|minor|major|critical",\n  "verdict": "layak|perlu_ditinjau",\n  "reason": "alasan singkat"\n}'
USER_PROMPT_TEMPLATE = 'Nilai kasus berikut.\n\n<case_id>{case_id}</case_id>\n<source_duri>{source_text}</source_duri>\n<shared_evidence>{shared_evidence}</shared_evidence>\n<candidate_translation>{candidate_text}</candidate_translation>'

PROMPT_HASH = hashlib.sha256(
    (
        SYSTEM_PROMPT
        + "\n"
        + USER_PROMPT_TEMPLATE
    ).encode()
).hexdigest()[:16]

if PROMPT_HASH != "d0e3d1938a7dcd05":
    raise RuntimeError(
        f"Prompt hash berubah: {PROMPT_HASH}"
    )

if (
    lock.get("benchmark_prompt_hash")
    != PROMPT_HASH
):
    raise RuntimeError(
        "Lock benchmark dan prompt main evaluator tidak identik."
    )

def build_user_prompt(row):
    candidate = str(
        row["candidate_text"]
    )
    return USER_PROMPT_TEMPLATE.format(
        case_id=row["eval_case_id"],
        source_text=row["source_text"],
        shared_evidence=row["shared_evidence"],
        candidate_text=(
            candidate
            if candidate.strip()
            else "[KOSONG]"
        ),
    )

print("PROMPT HASH:", PROMPT_HASH)
print("BENCHMARK PROMPT MATCH: PASS")

PROMPT HASH: d0e3d1938a7dcd05
BENCHMARK PROMPT MATCH: PASS


## 3. OpenRouter helper — benchmark-compatible

In [3]:
def headers():
    return {
        "Authorization":
            f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type":
            "application/json",
        "X-Title":
            APP_TITLE,
    }

def utc_now():
    return datetime.now(
        timezone.utc
    ).isoformat()

def normalize_content(content):
    if isinstance(content, str):
        return content

    if isinstance(content, list):
        out = []
        for item in content:
            if isinstance(item, str):
                out.append(item)
            elif isinstance(item, dict):
                text = (
                    item.get("text")
                    or item.get("content")
                )
                if text:
                    out.append(str(text))
        return "\n".join(out).strip()

    return str(content or "").strip()

def parse_json_object(text):
    raw = (
        str(text or "")
        .strip()
        .lstrip("\ufeff")
    )

    if not raw:
        raise ValueError(
            "Respons kosong."
        )

    try:
        obj = json.loads(raw)
        if not isinstance(obj, dict):
            raise ValueError(
                "JSON harus object."
            )
        return obj, True, raw

    except json.JSONDecodeError:
        pass

    decoder = json.JSONDecoder()

    for match in re.finditer(
        r"\{",
        raw,
    ):
        try:
            obj, end = decoder.raw_decode(
                raw[
                    match.start():
                ]
            )
        except json.JSONDecodeError:
            continue

        if isinstance(obj, dict):
            extracted = raw[
                match.start():
                match.start() + end
            ]
            return (
                obj,
                False,
                extracted,
            )

    raise ValueError(
        "Object JSON tidak ditemukan."
    )

def validate_prediction(pred):
    schema_errors = []
    logical_errors = []

    required = {
        "error_type",
        "severity",
        "verdict",
        "reason",
    }

    if not isinstance(pred, dict):
        return (
            False,
            False,
            ["Bukan JSON object."],
        )

    if required - set(pred):
        schema_errors.append(
            "Atribut hilang: "
            + str(
                sorted(
                    required
                    - set(pred)
                )
            )
        )

    if set(pred) - required:
        schema_errors.append(
            "Atribut tambahan: "
            + str(
                sorted(
                    set(pred)
                    - required
                )
            )
        )

    if pred.get(
        "error_type"
    ) not in ERROR_TYPES:
        schema_errors.append(
            "error_type invalid"
        )

    if pred.get(
        "severity"
    ) not in SEVERITIES:
        schema_errors.append(
            "severity invalid"
        )

    if pred.get(
        "verdict"
    ) not in VERDICTS:
        schema_errors.append(
            "verdict invalid"
        )

    if (
        not isinstance(
            pred.get("reason"),
            str,
        )
        or not pred.get(
            "reason",
            "",
        ).strip()
    ):
        schema_errors.append(
            "reason kosong/invalid"
        )

    schema_ok = (
        not schema_errors
    )

    if schema_ok:
        error_type = pred[
            "error_type"
        ]
        severity = pred[
            "severity"
        ]
        verdict = pred[
            "verdict"
        ]

        if (
            error_type == "correct"
            and (
                severity != "none"
                or verdict != "layak"
            )
        ):
            logical_errors.append(
                "correct harus none+layak"
            )

        if (
            error_type != "correct"
            and severity == "none"
        ):
            logical_errors.append(
                "non-correct tidak boleh none"
            )

        if (
            severity
            in {
                "major",
                "critical",
            }
            and verdict
            != "perlu_ditinjau"
        ):
            logical_errors.append(
                "major/critical harus perlu_ditinjau"
            )

        if (
            error_type
            == "empty_output"
            and (
                severity != "critical"
                or verdict
                != "perlu_ditinjau"
            )
        ):
            logical_errors.append(
                "empty_output harus critical+perlu_ditinjau"
            )

    return (
        schema_ok,
        (
            schema_ok
            and not logical_errors
        ),
        (
            schema_errors
            + logical_errors
        ),
    )

def request_retry(payload):
    retry_status = {
        408, 409, 425, 429,
        500, 502, 503, 504,
    }

    last_error = ""

    for attempt in range(
        1,
        MAX_HTTP_RETRIES + 1,
    ):
        try:
            response = requests.post(
                OPENROUTER_CHAT_URL,
                headers=headers(),
                json=payload,
                timeout=
                    REQUEST_TIMEOUT_SECONDS,
            )

            if response.status_code < 400:
                return response

            last_error = (
                f"HTTP {response.status_code}: "
                f"{response.text[:3000]}"
            )

            if (
                response.status_code
                not in retry_status
            ):
                raise RuntimeError(
                    last_error
                )

        except requests.RequestException as exc:
            last_error = (
                f"{type(exc).__name__}: {exc}"
            )

        if attempt < MAX_HTTP_RETRIES:
            time.sleep(
                min(
                    BASE_RETRY_DELAY_SECONDS
                    * (2 ** (attempt - 1)),
                    60.0,
                )
            )

    raise RuntimeError(
        last_error
        or "OpenRouter request gagal."
    )

def make_payload(panel_item, row):
    payload = {
        "model":
            panel_item["model_id"],
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content":
                    build_user_prompt(row),
            },
        ],
        "max_tokens":
            MAX_TOKENS,
        "stream":
            False,
        "provider": {
            "only":
                panel_item[
                    "provider_only"
                ],
            "allow_fallbacks":
                False,
            "require_parameters":
                True,
            "data_collection":
                "allow",
        },
    }

    # IMPORTANT:
    # Sama dengan benchmark V5:
    # parameter temperature TIDAK dikirim.
    if TEMPERATURE_SENT:
        raise RuntimeError(
            "TEMPERATURE_SENT harus False."
        )

    return payload

def call_once(panel_item, row):
    started = time.perf_counter()

    response = request_retry(
        make_payload(
            panel_item,
            row,
        )
    )

    latency_ms = (
        time.perf_counter()
        - started
    ) * 1000.0

    body = response.json()
    choices = body.get(
        "choices"
    ) or []

    if not choices:
        raise RuntimeError(
            "Respons tidak punya choices."
        )

    choice = choices[0]
    message = (
        choice.get("message")
        or {}
    )
    raw = normalize_content(
        message.get("content")
    )

    pred, strict, extracted = (
        parse_json_object(raw)
    )

    schema_ok, logical_ok, errors = (
        validate_prediction(pred)
    )

    usage = (
        body.get("usage")
        or {}
    )
    details = (
        usage.get(
            "completion_tokens_details"
        )
        or {}
    )

    return {
        "raw_response": raw,
        "extracted_json": extracted,
        "json_strict_valid":
            bool(strict),
        "schema_valid":
            bool(schema_ok),
        "logical_valid":
            bool(logical_ok),
        "validation_errors":
            " | ".join(errors),
        "prediction":
            pred,
        "returned_model":
            body.get("model", ""),
        "returned_provider":
            body.get("provider", ""),
        "finish_reason":
            choice.get(
                "finish_reason",
                "",
            ),
        "latency_ms":
            float(latency_ms),
        "prompt_tokens":
            usage.get(
                "prompt_tokens"
            ),
        "completion_tokens":
            usage.get(
                "completion_tokens"
            ),
        "reasoning_tokens":
            details.get(
                "reasoning_tokens",
                usage.get(
                    "reasoning_tokens"
                ),
            ),
        "total_tokens":
            usage.get(
                "total_tokens"
            ),
        "reported_cost":
            usage.get(
                "cost",
                body.get("cost"),
            ),
        "response_id":
            body.get("id", ""),
    }

print("OPENROUTER HELPER: READY")

OPENROUTER HELPER: READY


## 4. RUN_ID + technical-attempt checkpoint

In [4]:
blind_hash = hashlib.sha256(
    BLIND_PATH.read_bytes()
).hexdigest()[:16]

run_payload = {
    "blind_hash":
        blind_hash,
    "benchmark_prompt_hash":
        PROMPT_HASH,
    "models": [
        {
            "model_id":
                x["model_id"],
            "provider_only":
                x["provider_only"],
        }
        for x in panel
    ],
    "max_tokens":
        MAX_TOKENS,
    "temperature_sent":
        TEMPERATURE_SENT,
    "benchmark_run_id":
        lock.get(
            "benchmark_run_id"
        ),
}

RUN_ID = hashlib.sha256(
    json.dumps(
        run_payload,
        sort_keys=True,
    ).encode()
).hexdigest()[:16]

RUN_DIR = (
    OUTPUT_ROOT
    / f"run_{RUN_ID}"
)
RUN_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ATTEMPTS_PATH = (
    RUN_DIR
    / "main_evaluator_technical_attempts.csv"
)

FINAL_PATH = (
    RUN_DIR
    / "main_evaluator_final_477.csv"
)

MANIFEST_PATH = (
    RUN_DIR
    / "main_evaluator_run_manifest.json"
)

ATTEMPT_COLUMNS = [
    "timestamp_utc",
    "run_id",
    "eval_case_id",
    "panel_position",
    "evaluator_key",
    "display_name",
    "requested_model",
    "requested_provider",
    "technical_attempt",
    "api_success",
    "raw_response",
    "extracted_json",
    "pred_error_type",
    "pred_severity",
    "pred_verdict",
    "pred_reason",
    "json_strict_valid",
    "schema_valid",
    "logical_valid",
    "validation_errors",
    "returned_model",
    "returned_provider",
    "finish_reason",
    "latency_ms",
    "prompt_tokens",
    "completion_tokens",
    "reasoning_tokens",
    "total_tokens",
    "reported_cost",
    "response_id",
    "error_class",
    "error_message",
]

def load_attempts():
    if not ATTEMPTS_PATH.exists():
        return pd.DataFrame(
            columns=
                ATTEMPT_COLUMNS
        )

    return pd.read_csv(
        ATTEMPTS_PATH,
        dtype={
            "eval_case_id": str,
            "requested_model": str,
        },
        keep_default_na=False,
        encoding="utf-8-sig",
    )

def append_attempt(record):
    pd.DataFrame(
        [
            {
                c: record.get(c, "")
                for c in ATTEMPT_COLUMNS
            }
        ]
    ).to_csv(
        ATTEMPTS_PATH,
        mode="a",
        header=
            not ATTEMPTS_PATH.exists(),
        index=False,
        encoding="utf-8-sig",
    )

def successful_keys():
    attempts = load_attempts()

    if attempts.empty:
        return set()

    ok = (
        attempts[
            "api_success"
        ]
        .astype(str)
        .str.casefold()
        .eq("true")
        &
        attempts[
            "schema_valid"
        ]
        .astype(str)
        .str.casefold()
        .eq("true")
        &
        attempts[
            "logical_valid"
        ]
        .astype(str)
        .str.casefold()
        .eq("true")
    )

    return {
        (
            str(r.eval_case_id),
            str(r.requested_model),
        )
        for r in attempts[
            ok
        ].itertuples(
            index=False
        )
    }

print("RUN_ID :", RUN_ID)
print("RUN_DIR:", RUN_DIR)
print(
    "Already valid:",
    len(successful_keys()),
)

RUN_ID : f23f6109bec1a27d
RUN_DIR: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/llm_evaluator_main/panel_runs/run_f23f6109bec1a27d
Already valid: 0


## 5. Run 159 × 3 = 477 valid evaluator judgments

In [5]:
TARGET_VALID = (
    len(blind_df)
    * len(panel)
)

valid_before = len(
    successful_keys()
)

progress = tqdm(
    total=
        TARGET_VALID
        - valid_before,
    desc="Evaluator valid 477",
)

for _, row in blind_df.iterrows():

    for panel_item in panel:

        key = (
            str(
                row["eval_case_id"]
            ),
            str(
                panel_item[
                    "model_id"
                ]
            ),
        )

        if key in successful_keys():
            continue

        existing = load_attempts()

        if existing.empty:
            previous_count = 0
        else:
            previous_count = len(
                existing[
                    (
                        existing[
                            "eval_case_id"
                        ]
                        == key[0]
                    )
                    &
                    (
                        existing[
                            "requested_model"
                        ]
                        == key[1]
                    )
                ]
            )

        got_valid = False

        for local_try in range(
            1,
            MAX_TECHNICAL_VALIDATION_ATTEMPTS + 1,
        ):
            technical_attempt = (
                previous_count
                + local_try
            )

            record = {
                "timestamp_utc":
                    utc_now(),
                "run_id":
                    RUN_ID,
                "eval_case_id":
                    row[
                        "eval_case_id"
                    ],
                "panel_position":
                    panel_item[
                        "panel_position"
                    ],
                "evaluator_key":
                    panel_item[
                        "evaluator_key"
                    ],
                "display_name":
                    panel_item[
                        "display_name"
                    ],
                "requested_model":
                    panel_item[
                        "model_id"
                    ],
                "requested_provider":
                    "|".join(
                        panel_item[
                            "provider_only"
                        ]
                    ),
                "technical_attempt":
                    technical_attempt,
                "api_success":
                    False,
                "json_strict_valid":
                    False,
                "schema_valid":
                    False,
                "logical_valid":
                    False,
            }

            try:
                result = call_once(
                    panel_item,
                    row,
                )

                pred = result[
                    "prediction"
                ]

                record.update({
                    "api_success":
                        True,
                    "raw_response":
                        result[
                            "raw_response"
                        ],
                    "extracted_json":
                        result[
                            "extracted_json"
                        ],
                    "pred_error_type":
                        pred.get(
                            "error_type",
                            "",
                        ),
                    "pred_severity":
                        pred.get(
                            "severity",
                            "",
                        ),
                    "pred_verdict":
                        pred.get(
                            "verdict",
                            "",
                        ),
                    "pred_reason":
                        pred.get(
                            "reason",
                            "",
                        ),
                    "json_strict_valid":
                        result[
                            "json_strict_valid"
                        ],
                    "schema_valid":
                        result[
                            "schema_valid"
                        ],
                    "logical_valid":
                        result[
                            "logical_valid"
                        ],
                    "validation_errors":
                        result[
                            "validation_errors"
                        ],
                    "returned_model":
                        result[
                            "returned_model"
                        ],
                    "returned_provider":
                        result[
                            "returned_provider"
                        ],
                    "finish_reason":
                        result[
                            "finish_reason"
                        ],
                    "latency_ms":
                        result[
                            "latency_ms"
                        ],
                    "prompt_tokens":
                        result[
                            "prompt_tokens"
                        ],
                    "completion_tokens":
                        result[
                            "completion_tokens"
                        ],
                    "reasoning_tokens":
                        result[
                            "reasoning_tokens"
                        ],
                    "total_tokens":
                        result[
                            "total_tokens"
                        ],
                    "reported_cost":
                        result[
                            "reported_cost"
                        ],
                    "response_id":
                        result[
                            "response_id"
                        ],
                    "error_class":
                        "",
                    "error_message":
                        "",
                })

            except Exception as exc:
                record.update({
                    "error_class":
                        type(exc).__name__,
                    "error_message":
                        str(exc)[:3000],
                })

            append_attempt(
                record
            )

            technical_valid = (
                bool(
                    record.get(
                        "api_success",
                        False,
                    )
                )
                and bool(
                    record.get(
                        "schema_valid",
                        False,
                    )
                )
                and bool(
                    record.get(
                        "logical_valid",
                        False,
                    )
                )
            )

            if technical_valid:
                got_valid = True
                progress.update(1)
                break

            time.sleep(
                SLEEP_BETWEEN_CALLS_SECONDS
            )

        if not got_valid:
            raise RuntimeError(
                "Gagal memperoleh response valid setelah "
                f"{MAX_TECHNICAL_VALIDATION_ATTEMPTS} technical attempts: "
                f"{key}"
            )

        time.sleep(
            SLEEP_BETWEEN_CALLS_SECONDS
        )

progress.close()

attempts = load_attempts()

valid_mask = (
    attempts[
        "api_success"
    ]
    .astype(str)
    .str.casefold()
    .eq("true")
    &
    attempts[
        "schema_valid"
    ]
    .astype(str)
    .str.casefold()
    .eq("true")
    &
    attempts[
        "logical_valid"
    ]
    .astype(str)
    .str.casefold()
    .eq("true")
)

valid = attempts[
    valid_mask
].copy()

# Jika ada >1 response valid akibat rerun manual, gunakan valid pertama
# per case × model dan jangan double-count.
valid["_attempt_num"] = pd.to_numeric(
    valid[
        "technical_attempt"
    ],
    errors="coerce",
)

valid = (
    valid
    .sort_values(
        [
            "eval_case_id",
            "requested_model",
            "_attempt_num",
        ],
        kind="stable",
    )
    .drop_duplicates(
        [
            "eval_case_id",
            "requested_model",
        ],
        keep="first",
    )
    .drop(
        columns=[
            "_attempt_num"
        ]
    )
    .reset_index(drop=True)
)

if len(valid) != TARGET_VALID:
    raise RuntimeError(
        f"Valid final harus 477, ditemukan {len(valid)}."
    )

valid.to_csv(
    FINAL_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    "VALID EVALUATOR JUDGMENTS:",
    len(valid),
)
print("FINAL:", FINAL_PATH)

Evaluator valid 477:   0%|          | 0/477 [00:00<?, ?it/s]

VALID EVALUATOR JUDGMENTS: 477
FINAL: /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/llm_evaluator_main/panel_runs/run_f23f6109bec1a27d/main_evaluator_final_477.csv


## 6. Manifest + final audit

In [6]:
model_counts = (
    valid
    .groupby(
        "requested_model"
    )
    .size()
)

manifest = {
    "created_at_utc":
        utc_now(),
    "run_id":
        RUN_ID,
    "cases":
        159,
    "evaluator_agents":
        3,
    "valid_judgments":
        477,
    "benchmark_run_id":
        lock.get(
            "benchmark_run_id"
        ),
    "benchmark_prompt_hash":
        PROMPT_HASH,
    "prompt_exact_match_with_benchmark":
        True,
    "reference_used":
        False,
    "scenario_identity_used":
        False,
    "automatic_metrics_used":
        False,
    "temperature_sent":
        False,
    "max_tokens":
        MAX_TOKENS,
    "provider_pinned":
        True,
    "provider_fallback":
        False,
    "panel":
        panel,
    "blind_input":
        str(BLIND_PATH),
    "technical_attempts":
        str(ATTEMPTS_PATH),
    "final_valid_judgments":
        str(FINAL_PATH),
}

MANIFEST_PATH.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

checks = {
    "valid_477":
        len(valid) == 477,
    "three_models":
        len(model_counts) == 3,
    "159_each":
        bool(
            (model_counts == 159).all()
        ),
    "prompt_hash_exact":
        PROMPT_HASH
        == "d0e3d1938a7dcd05",
    "temperature_not_sent":
        TEMPERATURE_SENT is False,
    "reference_not_used":
        True,
    "final_exists":
        FINAL_PATH.exists(),
    "manifest_exists":
        MANIFEST_PATH.exists(),
}

failed = [
    k for k, v in checks.items()
    if not bool(v)
]

display(
    pd.DataFrame(
        [
            {"check": k, "ok": bool(v)}
            for k, v in checks.items()
        ]
    )
)

if failed:
    raise RuntimeError(
        f"FINAL MAIN EVALUATOR AUDIT GAGAL: {failed}"
    )

print("\n" + "=" * 78)
print("BENCHMARK-LOCKED 3 EVALUATOR AGENTS × 159: COMPLETE")
print("=" * 78)
display(
    valid[
        [
            "display_name",
            "requested_model",
            "returned_provider",
        ]
    ]
    .groupby(
        [
            "display_name",
            "requested_model",
            "returned_provider",
        ],
        as_index=False,
    )
    .size()
)

,check,ok
0,valid_477,True
1,three_models,True
2,159_each,True
3,prompt_hash_exact,True
4,temperature_not_sent,True
5,reference_not_used,True
6,final_exists,True
7,manifest_exists,True



BENCHMARK-LOCKED 3 EVALUATOR AGENTS × 159: COMPLETE


,display_name,requested_model,returned_provider,size
0,Claude Sonnet 4.6,anthropic/claude-sonnet-4.6,Anthropic,159
1,GPT-5.2,openai/gpt-5.2,OpenAI,159
2,Gemini 3.1 Pro Preview,google/gemini-3.1-pro-preview,Google AI Studio,159
